# 🌙 South Pole Tiles - Complete Workflow Notebook

This notebook demonstrates the **complete** workflow for lunar surface feature segmentation on south pole imagery.

## What You Will Learn

1. **Processing individual PNG tiles** - No need to create tiles from raster
2. **Running inference on pre-trained model** - Classify each tile
3. **Generating visualizations** - Clear crater feature detection at 0.1 threshold
4. **Aggregating results** - Combine predictions from all tiles
5. **Creating summary reports** - Statistics and quality metrics

## South Pole Tile Format

Unlike the Marius Hills raster tiles (1024x512 rectangular), south pole tiles come as individual PNG files with standard aspect ratios.

## Prerequisites

```bash
# Ensure model weights exist
ls -la weights/best_trained.pth

# Check tile directory (optional - inference is skipped if pre-computed probabilities exist)
ls -la results/south_pole/tiles/ | head -20
```

## 1. Environment Setup

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path
import sys
import cv2

# Add project root to path
sys.path.insert(0, r'Moon-Recognition/lunar_segmentation')
from PIL import Image
from IPython.display import Image as IPImage, display

# Import custom modules
from lunar_segmentation.models.unet import SmallUNet
from lunar_segmentation.data.preprocessing import build_three_channel_input, CLASS_NAMES
from lunar_segmentation.inference.predictor import Predictor
from lunar_segmentation.training.trainer import Trainer, BCEDiceLoss
from lunar_segmentation.data.datasets import MoonTileDataset

print("Environment setup complete!")
print(f"CLASS_NAMES: {CLASS_NAMES}")

## 2. Data Loading

In [ ]:
# Configuration
BASE_DIR = Path(r'data/MR').resolve()
if not (BASE_DIR / 'tiles' / 'index.csv').exists() or not (BASE_DIR / 'weights' / 'best_trained.pth').exists():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'tiles' / 'index.csv').exists() and (candidate / 'weights' / 'best_trained.pth').exists():
            BASE_DIR = candidate.resolve()
            break

TILES_DIR = BASE_DIR / 'lunar_south_pole' / 'tiles'
OUTPUT_DIR = BASE_DIR / 'results' / 'south_pole' / 'inference'
MODEL_WEIGHTS = BASE_DIR / 'weights' / 'best_trained.pth'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Check existing pre-computed data
PROBS_DIR = BASE_DIR / 'results' / 'south_pole' / 'probabilities'
VIZ_DIR = BASE_DIR / 'results' / 'south_pole' / 'visualizations'

print(f"Device: {DEVICE}")
print(f"Model weights exists: {MODEL_WEIGHTS.exists()}")
print(f"Tiles directory exists: {TILES_DIR.exists()}")
print(f"Pre-computed probabilities directory exists: {PROBS_DIR.exists()}")

# Count available data
if TILES_DIR.exists():
    tile_files = sorted(TILES_DIR.glob('tile_*.png'))
    print(f"\nFound {len(tile_files)} source PNG tiles")
else:
    tile_files = []
    print("\nNo source tiles directory - will use pre-computed probabilities")

if PROBS_DIR.exists():
    existing_probs = sorted(PROBS_DIR.glob('*.npz'))
    # Filter out the stats file
    existing_probs = [p for p in existing_probs if 'stats' not in p.name]
    print(f"Found {len(existing_probs)} pre-computed probability files")
else:
    existing_probs = []

if VIZ_DIR.exists():
    existing_vizs = sorted(VIZ_DIR.glob('*_viz.png'))
    print(f"Found {len(existing_vizs)} pre-computed visualizations")
else:
    existing_vizs = []

# Decide workflow
SKIP_INFERENCE = len(existing_probs) > 0 and len(tile_files) == 0
print(f"\nSkip inference (use pre-computed): {SKIP_INFERENCE}")


## 3. Model Initialization

In [ ]:
print(f"Loading model from {MODEL_WEIGHTS}...")
model = SmallUNet(in_channels=3, num_classes=len(CLASS_NAMES))
model.eval()

# Create predictor (Predictor handles checkpoint loading + legacy key remap)
predictor = Predictor(
    model=model,
    weights_path=MODEL_WEIGHTS,
    device=DEVICE
)

print(f"Model loaded successfully on {DEVICE}!")
print(f"Number of parameters: {sum(p.numel() for p in model.parameters()):,}")

print(f"\nClasses detected:")
for i, name in enumerate(CLASS_NAMES):
    print(f"  - Class {i}: {name}")

## 3.1 Optional: Model Training

This section only runs if the weights file is missing and training data is available.

In [ ]:
import os
CURRENT_DIR = os.getcwd()

if not MODEL_WEIGHTS.exists():
    print("Model weights missing. Attempting to train...")
    # Find the real MR dataset root on this machine.
    BASE_DIR = Path(r'data/MR').resolve()
    if not (BASE_DIR / 'tiles' / 'index.csv').exists() or not (BASE_DIR / 'weights' / 'best_trained.pth').exists():
        for candidate in [Path.cwd(), *Path.cwd().parents]:
            if (candidate / 'tiles' / 'index.csv').exists() and (candidate / 'weights' / 'best_trained.pth').exists():
                BASE_DIR = candidate.resolve()
                break
    BASE_DIR = BASE_DIR.resolve()
    if BASE_DIR.exists():
        os.chdir(BASE_DIR)
        print(f"Changed working directory to: {BASE_DIR}")
    else:
        print(f"Skipping chdir: BASE_DIR does not exist -> {BASE_DIR}")
    data_index = BASE_DIR / 'tiles' / 'index.csv'
    if data_index.exists():
        import pandas as pd
        df = pd.read_csv(data_index)
        dataset = MoonTileDataset(df, augment=True)
        loader = torch.utils.data.DataLoader(dataset, batch_size=4, shuffle=True)

        optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
        criterion = BCEDiceLoss()
        trainer = Trainer(model, optimizer, criterion, device=DEVICE)

        print("Starting 1-epoch training loop...")
        loss = trainer.train_one_epoch(loader)
        print(f"Training complete.")

        # Save weights
        MODEL_WEIGHTS.parent.mkdir(parents=True, exist_ok=True)
        torch.save(model.state_dict(), MODEL_WEIGHTS)
        print(f"Weights saved to {MODEL_WEIGHTS}")
        model.eval()  # switch back to eval mode
    else:
        print("No training index found. Please ensure data is prepared if you want to train.")
else:
    print("Pre-trained weights found. Skipping training section.")

os.chdir(CURRENT_DIR)


## 4. Inference Functions

In [ ]:

def load_tile_and_predict(tile_path):
    """
    Loads a single monochromatic tile, preprocesses it into a 3-channel feature 
    tensor (Normalized intensity, CLAHE, and Sobel gradients) at 256x256 resolution, 
    and executes network forward inference.

    Args:
        tile_path (Path or str): Path to the target PNG image tile.

    Returns:
        dict: Containing execution state, image matrices, and raw prediction cubes.
    """
    try:
        # Load the original planetary tile in grayscale mode
        img_raw = cv2.imread(str(tile_path), cv2.IMREAD_GRAYSCALE)
        if img_raw is None:
            raise FileNotFoundError(f"Target tile image could not be loaded: {tile_path}")

        # Resize the tile to 256x256 to perfectly match model training constraints
        img_resized = cv2.resize(img_raw, (256, 256))

        # --- CONSTRUCTING THE MULTI-CHANNEL FEATURE REPRESENTATION ---
        # Channel 1: Linear pixel intensity normalization scaled between [0.0, 1.0]
        ch1 = img_resized / 255.0

        # Channel 2: Contrast Limited Adaptive Histogram Equalization to resolve polar shadows
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        img_clahe = clahe.apply(img_resized)
        ch2 = img_clahe / 255.0

        # Channel 3: Sobel spatial gradient filters to extract topographic rim discontinuities
        sobelx = cv2.Sobel(img_resized, cv2.CV_64F, 1, 0, ksize=3)
        sobely = cv2.Sobel(img_resized, cv2.CV_64F, 0, 1, ksize=3)
        img_sobel = np.sqrt(sobelx**2 + sobely**2)
        
        # Max-normalize the Sobel gradient map to enforce numerical distribution stability
        if img_sobel.max() > 0:
            ch3 = img_sobel / img_sobel.max()
        else:
            ch3 = img_sobel

        # Stack independent matrices along the channel axis to create a (256, 256, 3) block
        tile_stacked = np.stack([ch1, ch2, ch3], axis=-1)

        # Transpose dimensions from HWC to CHW format to satisfy PyTorch U-Net layer requirements
        tile_chw = np.transpose(tile_stacked, (2, 0, 1))

        # Execute model inference via the predictor instance
        prob_cube = predictor.predict(tile_chw)

        return {
            'path': tile_path,
            'probabilities': prob_cube,
            'image': ch1,  # Exporting normalized grayscale matrix for graphical visualization overlays
            'success': True,
            'error': None
        }
    except Exception as e:
        return {
            'path': tile_path,
            'probabilities': None,
            'image': None,
            'success': False,
            'error': str(e)
        }

def generate_tile_viz(display_img, prob_cube, class_names, output_path):
    """
    Generates a synchronized multi-panel evaluation layout displaying the original 
    lunar topography, continuous target probabilities, and filtered categorical masks.

    Args:
        display_img (ndarray): Normalized grayscale background matrix.
        prob_cube (ndarray): Continuous output probability cube from the network.
        class_names (list): Ordered strings identifying target morphological classes.
        output_path (Path or str): Export target destination path for the PNG plot.
    """
    n_classes = len(class_names)
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    fig.suptitle(f"Tile Inference Report: {Path(output_path).name}", fontsize=14)

    # Render the base monochromatic topographic background reference map
    axes[0, 0].imshow(display_img, cmap='gray', vmin=0, vmax=1)
    axes[0, 0].set_title('Original Image', fontsize=10)
    axes[0, 0].axis('off')

    # Row 1 (Panels 1-3): continuous probability contours rendered via viridis map
    for col_idx, class_idx in enumerate(range(min(3, n_classes))):
        ax = axes[0, col_idx + 1]
        prob_img = prob_cube[class_idx].astype(np.float32)
        prob_max = float(np.max(prob_img))
        prob_min = float(np.min(prob_img))
        norm_prob = (prob_img - prob_min) / (prob_max - prob_min + 1e-8)

        ax.imshow(display_img, cmap='gray', vmin=0, vmax=1, alpha=0.4)
        ax.imshow(norm_prob, cmap='viridis', alpha=0.6)
        ax.set_title(f'{class_names[class_idx]}\nmax={prob_max:.3f}', fontsize=9)
        ax.axis('off')

    # Row 2 (Panels 0-3): binary masks filtered using a strict evaluation threshold
    adjusted_threshold = 0.1
    for col_idx, class_idx in enumerate(range(min(4, n_classes))):
        ax = axes[1, col_idx]
        prob_img = prob_cube[class_idx].astype(np.float32)
        mask = (prob_img > adjusted_threshold).astype(np.uint8)
        prob_max = float(np.max(prob_img))
        prob_min = float(np.min(prob_img))

        # Handle empty classification channels using adaptive background rejection masks
        if mask.sum() == 0:
            if prob_max > prob_min:
                norm_prob = (prob_img - prob_min) / (prob_max - prob_min + 1e-8)
                ax.imshow(norm_prob, cmap='plasma', alpha=0.7)
                title = f'{class_names[class_idx]}\nmax={prob_max:.3f}\nNo features at {adjusted_threshold}'
            else:
                ax.imshow(np.zeros_like(display_img), cmap='gray')
                title = f'{class_names[class_idx]}\nNo detections'
        else:
            # Render confirmed detections using dedicated colormap layers
            if class_idx == 0:
                ax.imshow(mask, cmap='Reds', alpha=0.9)
            else:
                ax.imshow(mask, cmap='hot', alpha=0.8)
            title = f'{class_names[class_idx]}\n>{adjusted_threshold} | n={mask.sum()}'
        ax.set_title(title, fontsize=9)
        ax.axis('off')

    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.close()

def run_batch_inference(tiles_dir, predictor, output_dir, max_tiles=None):
    """
    Manages the batch inference execution pipeline over the dataset array, 
    isolating raw arrays from visual multi-panel inspection files.

    Args:
        tiles_dir (Path): Input source directory containing target image tiles.
        predictor (object): Compiled wrapper managing model evaluations.
        output_dir (Path): Root directory where subsets will be serialized.
        max_tiles (int, optional): Bound upper-limit termination for testing.

    Returns:
        list: Collection of dictionary records tracks batch status metrics.
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    probs_dir = output_dir / 'probabilities'
    viz_dir = output_dir / 'visualizations'
    probs_dir.mkdir(exist_ok=True)
    viz_dir.mkdir(exist_ok=True)

    # Sort and index target planetary tile files matching the naming format
    tile_files = sorted(tiles_dir.glob('tile_*.png'))
    if max_tiles:
        tile_files = tile_files[:max_tiles]
    print(f"Processing {len(tile_files)} tiles...")

    results = []
    for i, tile_path in enumerate(tile_files):
        result = load_tile_and_predict(tile_path)
        results.append(result)
        
        # Serialize raw outputs and plot files exclusively for nominal inference passes
        if result['success']:
            prob_cube = result['probabilities']
            img = result['image']
            
            # Non-volatile saving of continuous probability logs to probabilities/
            np.savez(probs_dir / tile_path.name, probabilities=prob_cube)
            
            # Export graphical verification layouts to visualizations/
            viz_path = viz_dir / f'{tile_path.stem}_viz.png'
            generate_tile_viz(img, prob_cube, CLASS_NAMES, viz_path)
            
    return results

print("Functions ready.")

## 5. Run Inference

In [ ]:

import os
from pathlib import Path
from IPython.display import Image as IPImage, display

# 1. Ensure the main output directory exists
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 2. Verify input files before starting execution
tile_files = sorted(TILES_DIR.glob('tile_*.png'))

if len(tile_files) > 0:
    print(f"Found {len(tile_files)} images in TILES_DIR. Starting inference batch...")
    
    # Run batch inference on the first 20 tiles as a quick pipeline integration test
    results = run_batch_inference(TILES_DIR, predictor, OUTPUT_DIR, max_tiles=3639)
    
    success_count = sum(1 for r in results if r.get('success', False))
    failure_count = len(results) - success_count
    
    # DEBUG BLOCK: If there are failures, print the internal exceptions to understand why
    if failure_count > 0:
        print("\n--- Debugging Failures ---")
        failed_records = [r for r in results if not r.get('success', False)]
        # Show the error message of the first 3 failed attempts
        for idx, f_rec in enumerate(failed_records[:3]):
            print(f"File: {Path(f_rec.get('path', 'Unknown')).name} | Error: {f_rec.get('error')}")
            
else:
    print(f"Error: No data found in target directory: {TILES_DIR}")
    success_count = 0
    failure_count = 0

print(f"\nSummary: {success_count} success, {failure_count} failure")

# 3. Synchronized visual verification check
print("\n--- Visual Verification Check ---")
viz_folder = OUTPUT_DIR / 'visualizations'
generated_plots = sorted(list(viz_folder.glob("*.png")))

if len(generated_plots) > 0:
    test_viz_file = generated_plots[0]
    print(f"Displaying sample analytical report generated by U-Net: {test_viz_file.name}")
    display(IPImage(filename=str(test_viz_file)))
else:
    print(f"Warning: No visualization plots found in {viz_folder}.")
    print("Please resolve the failures in load_tile_and_predict using the debug logs above.")

## 6. Aggregation

In [ ]:
def aggregate_results(probs_dir, class_names):
    """Stream through .npz files one at a time to avoid loading all tiles
    into RAM simultaneously.

    Instead of stacking (N, C, H, W) into one giant array we keep only:
      - sum_probs  (float64, shape C x H x W): running sum  -> /n  = mean
      - max_probs  (float32, shape C x H x W): element-wise max across tiles
      - global_max (float32, length C)        : per-class scalar max

    Peak extra RAM = 3 × (C × H × W) regardless of tile count.
    """
    import json
    prob_files = sorted([p for p in probs_dir.glob('*.npz') if 'stats' not in p.name])
    n = len(prob_files)
    if n == 0:
        print('No probability files found – nothing to aggregate.')
        return {}

    # --- initialise accumulators from the first file ---
    first_cube = np.load(prob_files[0])['probabilities']   # (C, H, W)
    n_classes  = first_cube.shape[0]
    sum_probs  = first_cube.astype(np.float64)             # accumulate in float64
    max_probs  = first_cube.copy()                         # element-wise max (C, H, W)
    global_max = np.array(
        [float(np.max(first_cube[i])) for i in range(n_classes)],
        dtype=np.float32)                                  # per-class scalar max
    del first_cube

    # --- stream the remaining files one at a time ---
    for f in prob_files[1:]:
        cube = np.load(f)['probabilities']                 # (C, H, W)
        sum_probs += cube.astype(np.float64)
        np.maximum(max_probs, cube, out=max_probs)         # in-place element-wise max
        for i in range(n_classes):
            cm = float(np.max(cube[i]))
            if cm > global_max[i]:
                global_max[i] = cm
        del cube

    avg_probs = (sum_probs / n).astype(np.float32)
    del sum_probs

    stats = {
        'tile_count': n,
        'class_names': class_names,
        'avg_max_per_class':    {c: float(np.max(avg_probs[i]))     for i, c in enumerate(class_names)},
        'global_max_per_class': {c: float(global_max[i])            for i, c in enumerate(class_names)},
        'features_detected':    {c: int((max_probs[i] > 0.1).sum()) for i, c in enumerate(class_names)},
        '_avg_probs': avg_probs,
    }

    # Save stats.json (skip private keys that start with '_')
    json_path = probs_dir.parent / 'stats.json'
    with open(json_path, 'w') as fh:
        json.dump({k: v for k, v in stats.items() if not k.startswith('_')}, fh, indent=2)
    print(f'Stats saved to {json_path}')
    return stats

# Modified path specifying the 'probabilities' subfolder
SOTTO_CARTELLA_NPZ = OUTPUT_DIR / 'probabilities'
print(f"Verifico la presenza di file .npz in: {SOTTO_CARTELLA_NPZ}")
stats = aggregate_results(SOTTO_CARTELLA_NPZ, CLASS_NAMES)
print("Aggregation complete.")

## 7. Visualization

In [ ]:
SOTTO_CARTELLA_VIZ = OUTPUT_DIR / 'visualizations'
viz_files = sorted(SOTTO_CARTELLA_VIZ.glob('*_viz.png'))
print(f"Displaying {len(viz_files)} tiles...")

for i, viz_file in enumerate(viz_files[:10]):
    print(f"\n--- Tile {i+1}: {viz_file.name} ---")
    display(IPImage(filename=str(viz_file)))


In [ ]:
print("=== QUALITY ASSESSMENT ===")

# Sincronizziamo il percorso con dove la parte 6 ha salvato il file
# Il file stats.json si trova dentro la cartella principale dei risultati (OUTPUT_DIR)
stats_path = OUTPUT_DIR / 'stats.json'

# Inizializziamo i dizionari principali
avg_max = {}
detected = {}

# 1. Proviamo a prendere i dati dalla variabile in memoria 'stats'
if 'stats' in locals() and stats:
    avg_max = stats.get('avg_max_per_class', {})
    detected = stats.get('features_detected', {})

# 2. Se la memoria è vuota, leggiamo il file stats.json UNA SOLA VOLTA prima del ciclo
if (not avg_max or not detected) and stats_path.exists():
    print(f"Loading cached metrics from {stats_path.name}...")
    saved_stats = json.loads(stats_path.read_text())
    avg_max = saved_stats.get('avg_max_per_class', {})
    detected = saved_stats.get('features_detected', {})

# 3. Ciclo di stampa finale per tutte le classi morfologiche
for c in CLASS_NAMES:
    if c in avg_max and c in detected:
        print(f"{c}: MaxProb={avg_max.get(c, 0.0):.4f}, Detections={detected.get(c, 0)}")
    else:
        print(f"{c}: no aggregation stats available")

## 10. Summary
Notebook completed successfully!